# 09 - Local end-to-end verification

- **Meeting item:** Local workflow
- **Commit:** C12 - `test: add end-to-end plan compliance workflow`
- **Commit achievement:** proves one local command executes checks and notebooks.
- **Data mode:** `synthetic`
- **Evidence labels used:** verified
- **Human gate:** can a user run local verification and get PASS / INCONCLUSIVE / BLOCKED outputs?
- **Stop condition:** only CI works, or generated outputs pollute the source tree.

This notebook does not train models. It calls `scripts/verify_repository.py`, which is the same
command `make verify` ends with. Every check returns one of three statuses. INCONCLUSIVE means
evidence is missing, not that a rule passed.

## 1. Local command (evidence label: verified)

The primary gate is local. CI runs the same steps so a reviewer can see the result without a
checkout; it does not invent checks that only exist remotely.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

import p22

VERIFY = p22.REPO_ROOT / "scripts" / "verify_repository.py"
CI = p22.REPO_ROOT / ".github" / "workflows" / "ci.yml"
E2E = p22.REPO_ROOT / "tests" / "test_end_to_end.py"

print("verify script:", VERIFY.is_file(), VERIFY.relative_to(p22.REPO_ROOT))
print("CI workflow:", CI.is_file(), CI.relative_to(p22.REPO_ROOT))
print("end-to-end test:", E2E.is_file(), E2E.relative_to(p22.REPO_ROOT))
print("approval blocked:", p22.approval_blocked())
print("data mode:", p22.SYNTHETIC)
print()
print("local commands after C12:")
print("  make verify")
print("  make notebook-check")
print("  .venv-p22/bin/python scripts/verify_repository.py")

verify script: True scripts/verify_repository.py
CI workflow: True .github/workflows/ci.yml
end-to-end test: True tests/test_end_to_end.py
approval blocked: True
data mode: synthetic

local commands after C12:
  make verify
  make notebook-check
  .venv-p22/bin/python scripts/verify_repository.py


## 2. Run the verifier (evidence label: verified)

`--allow-dirty` is used here so this notebook can run while its own commit is still open.
Without the flag, an uncommitted tree is BLOCKED. After the commit lands on a clean tree,
the same command without the flag is what `make verify` expects.

In [2]:
completed = subprocess.run(
    [sys.executable, str(VERIFY), "--allow-dirty", "--json"],
    cwd=p22.REPO_ROOT,
    capture_output=True,
    text=True,
    check=False,
)
assert completed.stdout, completed.stderr[-2000:]
payload = json.loads(completed.stdout)

print(f"exit code: {completed.returncode}")
print(f"verdict: {payload['verdict']}")
print()
width = max(len(check["name"]) for check in payload["checks"])
for check in payload["checks"]:
    print(f"{check['status']:<13} {check['name']:<{width}}  {check['detail']}")

exit code: 3
verdict: INCONCLUSIVE

PASS          plan guard status                                     plan guard runs; 13 of 13 steps complete, next action awaiting_professor_approval
PASS          python sources parse                                  65 files parse
PASS          test suite collects                                   418 tests collected, 31 in the end-to-end module
PASS          end-to-end test marked slow and integration           slow and integration markers present
PASS          run configuration                                     synthetic, five seeds, six baselines, cpu, limitations stated
PASS          intervention set                                      7 held-out interventions defined
PASS          report schema                                         reports carry provenance, performance, routing, interventions, limitations, approval
PASS          legacy evidence unchanged                             manifest verifies and 10 legacy or experiment files are u

## 3. Rules the verifier enforces (evidence label: verified)

Each row below is one named check. A blocked check fails the whole run. An inconclusive check
means something has not been produced yet (for example, executed notebooks before
`make notebook-check`).

In [3]:
import pandas as pd

rows = [
    {
        "check": check["name"],
        "status": check["status"],
        "detail": check["detail"],
    }
    for check in payload["checks"]
]
table = pd.DataFrame(rows)
display(table)

blocked = [row for row in rows if row["status"] == "BLOCKED"]
inconclusive = [row for row in rows if row["status"] == "INCONCLUSIVE"]
passed = [row for row in rows if row["status"] == "PASS"]
print(f"PASS: {len(passed)}  INCONCLUSIVE: {len(inconclusive)}  BLOCKED: {len(blocked)}")
assert not blocked, "blocked checks: " + "; ".join(row["check"] for row in blocked)

,check,status,detail
0,plan guard status,PASS,"plan guard runs; 13 of 13 steps complete, next..."
1,python sources parse,PASS,65 files parse
2,test suite collects,PASS,"418 tests collected, 31 in the end-to-end module"
3,end-to-end test marked slow and integration,PASS,slow and integration markers present
4,run configuration,PASS,"synthetic, five seeds, six baselines, cpu, lim..."
5,intervention set,PASS,7 held-out interventions defined
6,report schema,PASS,"reports carry provenance, performance, routing..."
7,legacy evidence unchanged,PASS,manifest verifies and 10 legacy or experiment ...
8,no real dataset files tracked,PASS,no tracked file carries a dataset suffix (.h5a...
9,no condition-specific work while approval is b...,PASS,none of the 11 condition terms appear in scann...


PASS: 12  INCONCLUSIVE: 1  BLOCKED: 0


## 4. Approval boundary and generated outputs (evidence label: verified)

Generated notebooks and reports live under `reports/generated/`, which is gitignored. Source
notebooks under `notebooks/implementation/` are never overwritten by verification.

In [4]:
from p22.eval.faithfulness import INTERVENTIONS
from p22.reports.render import SECTION_TITLES

generated = p22.REPO_ROOT / "reports" / "generated"
ignore = subprocess.run(
    ["git", "-C", str(p22.REPO_ROOT), "check-ignore", "reports/generated"],
    capture_output=True,
    text=True,
    check=False,
)
tracked_generated = subprocess.run(
    ["git", "-C", str(p22.REPO_ROOT), "ls-files", "reports/generated"],
    capture_output=True,
    text=True,
    check=True,
).stdout.split()

print("reports/generated ignored:", ignore.returncode == 0)
print("tracked under reports/generated:", tracked_generated or "<none>")
print("interventions defined:", len(INTERVENTIONS))
print("report sections:", list(SECTION_TITLES))
print("approvals:", p22.load_approvals()["approval_state"])

by_name = {check["name"]: check for check in payload["checks"]}
for name in (
    "no real dataset files tracked",
    "no condition-specific work while approval is blocked",
    "legacy evidence unchanged",
    "generated outputs stay out of the source tree",
    "run configuration",
    "intervention set",
    "report schema",
):
    check = by_name[name]
    print(f"{check['status']}: {name}")

reports/generated ignored: True
tracked under reports/generated: <none>
interventions defined: 7
report sections: ['Provenance', 'Predictive performance', 'Routing signals', 'Intervention evidence', 'Limitations', 'Approval and next step']
approvals: blocked
PASS: no real dataset files tracked
PASS: no condition-specific work while approval is blocked
PASS: legacy evidence unchanged
PASS: generated outputs stay out of the source tree
PASS: run configuration
PASS: intervention set
PASS: report schema


## 5. Clean-tree rule without the flag (evidence label: verified)

A dirty working tree is not a pass. The flag only softens the report while a step is still
open. After commit, `make verify` runs without it.

In [5]:
import importlib.util

spec = importlib.util.spec_from_file_location("verify_repository_nb", VERIFY)
verifier = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = verifier
spec.loader.exec_module(verifier)

soft = verifier.check_repository_clean(allow_dirty=True)
hard = verifier.check_repository_clean(allow_dirty=False)
print(f"with --allow-dirty: {soft.status} - {soft.detail}")
print(f"without flag:       {hard.status} - {hard.detail}")
assert soft.status in (verifier.PASS, verifier.INCONCLUSIVE)
assert hard.status in (verifier.PASS, verifier.BLOCKED)
if hard.status == verifier.BLOCKED:
    print("expected before C12 commit: dirty tree blocks the clean check")
else:
    print("working tree is clean; make verify can return PASS on this check")

with --allow-dirty: INCONCLUSIVE - 4 uncommitted path(s); allowed by --allow-dirty
without flag:       BLOCKED - 4 uncommitted path(s)
expected before C12 commit: dirty tree blocks the clean check


## 6. Human gate

Answer before moving on: can a user run local verification and get PASS / INCONCLUSIVE /
BLOCKED outputs?

In [6]:
ci_text = CI.read_text()
checks = {
    "verify script exists and runs": VERIFY.is_file() and bool(payload["checks"]),
    "every check returns PASS, INCONCLUSIVE, or BLOCKED": all(
        check["status"] in ("PASS", "INCONCLUSIVE", "BLOCKED") for check in payload["checks"]
    ),
    "verdict is one of the three statuses": payload["verdict"] in (
        "PASS",
        "INCONCLUSIVE",
        "BLOCKED",
    ),
    "nothing is blocked while this step is open": not blocked,
    "CI workflow runs the same local make targets": CI.is_file()
    and "make plan-status" in ci_text
    and "make test-all" in ci_text
    and "scripts/verify_repository.py --execute-notebooks" in ci_text
    and "scripts/verify_repository.py --json" in ci_text,
    "end-to-end test is marked slow and integration": by_name[
        "end-to-end test marked slow and integration"
    ]["status"]
    == "PASS",
    "generated outputs are ignored and untracked": by_name[
        "generated outputs stay out of the source tree"
    ]["status"]
    == "PASS",
    "no real dataset files are tracked": by_name["no real dataset files tracked"]["status"]
    == "PASS",
    "approval remains blocked": p22.approval_blocked(),
    "seven interventions and six report sections are declared": len(INTERVENTIONS) == 7
    and len(SECTION_TITLES) == 6,
}
for description, passed_flag in checks.items():
    print(f"{'PASS' if passed_flag else 'FAIL'}: {description}")
assert all(checks.values()), "C12 gate not satisfied"
print()
print("C12 gate: PASS")

PASS: verify script exists and runs
PASS: every check returns PASS, INCONCLUSIVE, or BLOCKED
PASS: verdict is one of the three statuses
PASS: nothing is blocked while this step is open
PASS: CI workflow runs the same local make targets
PASS: end-to-end test is marked slow and integration
PASS: generated outputs are ignored and untracked
PASS: no real dataset files are tracked
PASS: approval remains blocked
PASS: seven interventions and six report sections are declared

C12 gate: PASS


## Gate

**PASS**

- **Observed:** `scripts/verify_repository.py` returns PASS / INCONCLUSIVE / BLOCKED for each
  named check; CI runs the same local make targets; generated outputs stay under the ignored
  `reports/generated/` tree; no real dataset files are tracked; approval remains blocked.
- **Remains unknown:** whether a clean post-commit `make verify` returns an overall PASS on
  this machine until the C12 commit lands and notebooks are re-executed; the approved dataset
  and target remain undecided.
- **Next decision owner:** researcher continues with C13 (methods protocol and handoff).
  Condition-specific work stays blocked pending Professor Fang.